In [ ]:
!pip install -q diffusers transformers accelerate peft

In [ ]:
!git clone https://github.com/skminhazuddin/SixthSemesterProjects.git

fatal: destination path 'SixthSemesterProjects' already exists and is not an empty directory.


In [ ]:
# ================= SAFETY INIT (FIX FOR db ERROR) =================
import os
import sqlite3
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from diffusers import StableDiffusionPipeline, DDPMScheduler
import torch.nn.functional as F
from torch.optim import AdamW
from peft import get_peft_model, LoraConfig
import torchvision.transforms as T

BASE_PATH = "/content/SixthSemesterProjects/resource"
DB_PATH = "/content/training_data.db"

# ================= DATABASE =================
class DatabaseManager:
    def __init__(self):
        self.conn = sqlite3.connect(DB_PATH)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS images (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_path TEXT UNIQUE,
            description TEXT,
            tags TEXT,
            category TEXT,
            character TEXT,
            trained INTEGER DEFAULT 0
        )
        """)
        self.conn.commit()

    def insert_image(self, path, description, tags, category, character):
        cursor = self.conn.cursor()
        cursor.execute("SELECT id FROM images WHERE image_path=?", (path,))
        if cursor.fetchone() is None:
            cursor.execute("""
            INSERT INTO images (image_path, description, tags, category, character, trained)
            VALUES (?,?,?,?,?,0)
            """, (path, description, tags, category, character.lower()))
            self.conn.commit()

    def get_untrained_images(self, category):
        cursor = self.conn.cursor()
        cursor.execute("""
        SELECT id, image_path, description, tags, character
        FROM images WHERE trained=0 AND category=?
        """, (category,))
        return cursor.fetchall()

    def get_all_characters(self):
        cursor = self.conn.cursor()
        cursor.execute("SELECT DISTINCT character FROM images WHERE character!=''")
        return [row[0] for row in cursor.fetchall()]

# ================= DATASET =================
class DatasetManager:
    def __init__(self, db):
        self.db = db

    def parse_txt(self, txt_path):
        description, tags, character = "", "", ""
        with open(txt_path, "r", encoding="utf-8") as f:
            content = f.read()

        if "Description:" in content:
            description = content.split("Description:")[1].split("Tags:")[0].strip()

        if "Tags:" in content:
            tags = content.split("Tags:")[1].strip()

        if "Character:" in content:
            character = content.split("Character:")[1].strip().lower()

        return description, tags, character

    def process_folder(self, folder, category):
        for root_dir, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith((".jpg", ".png", ".jpeg")):
                    img_path = os.path.join(root_dir, file)
                    txt_path = os.path.splitext(img_path)[0] + ".txt"

                    description, tags, character = "", "", ""
                    if os.path.exists(txt_path):
                        description, tags, character = self.parse_txt(txt_path)

                    self.db.insert_image(img_path, description, tags, category, character)

    def load_all_data(self):
        self.process_folder(BASE_PATH + "/HandaBhondaComicFigures", "character")
        self.process_folder(BASE_PATH + "/Comic_Scenes", "scene")

# ================= DATASET CLASS =================
class ImageDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.transform = T.Compose([
            T.Resize((512, 512)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_id, img_path, description, tags, character = self.data[idx]
        caption = f"{character}. {description}. {tags}" if character else f"{description}. {tags}"

        img = Image.open(img_path).convert("RGB")
        return {
            "image_id": image_id,
            "pixel_values": self.transform(img),
            "caption": caption
        }

# ================= TRAINER =================
class ModelTrainer:
    def __init__(self, dataset_manager, output_dir):
        self.dataset_manager = dataset_manager
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5"
        ).to(self.device)

        self.unet = get_peft_model(
            self.pipe.unet,
            LoraConfig(r=8, lora_alpha=32, target_modules=["to_q", "to_v"])
        )

        self.vae = self.pipe.vae
        self.text_encoder = self.pipe.text_encoder
        self.tokenizer = self.pipe.tokenizer
        self.scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
        )

    def train(self, category):
        data = self.dataset_manager.db.get_untrained_images(category)
        if not data:
            print("No data found!")
            return

        loader = DataLoader(ImageDataset(data), batch_size=2, shuffle=True)
        optimizer = AdamW(self.unet.parameters(), lr=5e-5)

        for step, batch in enumerate(loader):
            pixel_values = batch["pixel_values"].to(self.device)
            captions = batch["caption"]

            input_ids = self.tokenizer(
                list(captions), padding="max_length",
                truncation=True, max_length=77,
                return_tensors="pt"
            ).input_ids.to(self.device)

            latents = self.vae.encode(pixel_values).latent_dist.sample() * 0.18215
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, 1000, (latents.shape[0],), device=self.device)

            noisy = self.scheduler.add_noise(latents, noise, timesteps)
            hidden = self.text_encoder(input_ids)[0]

            pred = self.unet(noisy, timesteps, hidden).sample
            loss = F.mse_loss(pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"Step {step} Loss {loss.item():.4f}")

# ================= INIT (FIXED ORDER) =================
db = DatabaseManager()
dataset = DatasetManager(db)
dataset.load_all_data()

print("Characters found:", db.get_all_characters())

# ================= GENERATOR =================
class PromptGenerator:
    def __init__(self, db):
        self.db = db
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            safety_checker=None
        ).to(self.device)

    def detect_character(self, prompt):
        chars = sorted(self.db.get_all_characters(), key=len, reverse=True)
        for c in chars:
            if c in prompt.lower():
                return c
        return None

    def generate(self, prompt):
        char = self.detect_character(prompt)
        final = f"{char}, {prompt}" if char else prompt

        with torch.no_grad():
            return self.pipe(final).images[0]

# ================= UI =================
import ipywidgets as widgets
from IPython.display import display, clear_output

generator = PromptGenerator(db)

btn_train_char = widgets.Button(description="Train Characters", button_style='success')
btn_train_scene = widgets.Button(description="Train Scenes", button_style='info')
btn_generate = widgets.Button(description="Generate Image", button_style='primary')

prompt_box = widgets.Text(
    placeholder='Enter prompt...',
    description='Prompt:',
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()

def train_char(b):
    with output:
        clear_output()
        ModelTrainer(dataset, "lora_character").train("character")

def train_scene(b):
    with output:
        clear_output()
        ModelTrainer(dataset, "lora_scene").train("scene")

def generate(b):
    with output:
        clear_output()
        img = generator.generate(prompt_box.value)
        display(img)

btn_train_char.on_click(train_char)
btn_train_scene.on_click(train_scene)
btn_generate.on_click(generate)

display(widgets.VBox([
    widgets.HTML("<h2>🎨 Comic AI Generator</h2>"),
    btn_train_char,
    btn_train_scene,
    prompt_box,
    btn_generate,
    output
]))

In [ ]:
display(widgets.VBox([
    widgets.HTML("<h2>🎨 Comic AI Generator</h2>"),
    btn_train_char,
    btn_train_scene,
    prompt_box,
    btn_generate,
    output
]))